#  Data Cleaning

**Objective:** Take the extracted master dataset, handle missing values, fix data types, standardise formats, drop irrelevant columns, and output a clean, analysis-ready file to `data/processed/`.  

**Input:** `data/extracted/master_extracted.csv`  , `data/extracted/modern_pit_stops.csv`  , `data/extracted/modern_constructor_standings.csv` 
 
**Output:** `data/processed/f1_modern_clean.csv`  , `data/processed/pit_stops_clean.csv`  , `data/processed/constructor_standings_clean.csv`


## 1. Setup & Imports


In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


## 2. Load Extracted Data


In [2]:
EXTRACT_DIR = '../data/extracted/'

master = pd.read_csv(os.path.join(EXTRACT_DIR, 'master_extracted.csv'))
pit_stops = pd.read_csv(os.path.join(EXTRACT_DIR, 'modern_pit_stops.csv'))
standings = pd.read_csv(os.path.join(EXTRACT_DIR, 'modern_constructor_standings.csv'))

print(f'Master:     {master.shape}')
print(f'Pit stops:  {pit_stops.shape}')
print(f'Standings:  {standings.shape}')


Master:     (4626, 35)
Pit stops:  (8360, 7)
Standings:  (2319, 7)


## 3. Pre-Cleaning Snapshot
Before making any changes, we log the current state of the data. Every transformation step is documented below so the pipeline is fully reproducible.


In [3]:
# Log the initial state
cleaning_log = []

def log_step(step_name, description, rows_before, rows_after, cols_before=None, cols_after=None):
    """Record each cleaning step for the final report."""
    entry = {
        'Step': step_name,
        'Description': description,
        'Rows Before': rows_before,
        'Rows After': rows_after,
        'Rows Removed': rows_before - rows_after,
    }
    if cols_before is not None:
        entry['Cols Before'] = cols_before
        entry['Cols After'] = cols_after
    cleaning_log.append(entry)
    print(f'[{step_name}] {description} | {rows_before} -> {rows_after} rows')

print('------PRE-CLEANING STATE-------')
print(f'Shape: {master.shape}')
print()
print('Null counts per column:')
null_counts = master.isna().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False).to_string())


------PRE-CLEANING STATE-------
Shape: (4626, 35)

Null counts per column:
time               2219
milliseconds       2219
position            721
fastestLap          217
fastestLapTime      217
fastestLapSpeed     217


## 4. Step 1 :  Drop Irrelevant Columns
Several columns are either redundant, too sparse, or not useful for our Investment ROI analysis.


In [4]:
rows_before = len(master)
cols_before = master.shape[1]

# Columns to drop and reasons:
drop_cols = {
    'number': 'Car number — redundant for constructor-level analysis',
    'position': '40% null — use positionOrder instead (always filled)',
    'positionText': 'String version of positionOrder — redundant',
    'time': '71% null — only race winner gets absolute time',
    'milliseconds': '71% null — same as above',
    'fastestLap': '69% null — sparse, not relevant to Investment KPI',
    'rank': '68% null — fastest lap rank, not relevant',
    'fastestLapTime': '69% null — sparse',
    'fastestLapSpeed': '69% null — sparse',
    'dob': 'Driver date of birth — not relevant to constructor ROI',
}

print('Columns being dropped:')
for col, reason in drop_cols.items():
    if col in master.columns:
        print(f'  {col:<20} -> {reason}')

master = master.drop(columns=[c for c in drop_cols.keys() if c in master.columns])

log_step('Drop Columns', f'Removed {cols_before - master.shape[1]} sparse/irrelevant columns',
         rows_before, len(master), cols_before, master.shape[1])

print(f'\nRemaining columns ({master.shape[1]}): {list(master.columns)}')


Columns being dropped:
  number               -> Car number — redundant for constructor-level analysis
  position             -> 40% null — use positionOrder instead (always filled)
  positionText         -> String version of positionOrder — redundant
  time                 -> 71% null — only race winner gets absolute time
  milliseconds         -> 71% null — same as above
  fastestLap           -> 69% null — sparse, not relevant to Investment KPI
  rank                 -> 68% null — fastest lap rank, not relevant
  fastestLapTime       -> 69% null — sparse
  fastestLapSpeed      -> 69% null — sparse
  dob                  -> Driver date of birth — not relevant to constructor ROI
[Drop Columns] Removed 10 sparse/irrelevant columns | 4626 -> 4626 rows

Remaining columns (25): ['resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'points', 'laps', 'statusId', 'year', 'round', 'circuitId', 'race_name', 'race_date', 'constructor_name', 'constructor_nationality', 'for

## 5. Step 2 : Fix Data Types
Ensure columns have the correct types for analysis.


In [5]:
rows_before = len(master)

# Convert race_date to datetime
master['race_date'] = pd.to_datetime(master['race_date'], format='%Y-%m-%d')

# Ensure numeric columns are the right type
master['points'] = master['points'].astype(float)
master['grid'] = master['grid'].astype(int)
master['positionOrder'] = master['positionOrder'].astype(int)
master['laps'] = master['laps'].astype(int)
master['year'] = master['year'].astype(int)

print('Data type conversions applied:')
print(master.dtypes.to_string())
print()

log_step('Fix Dtypes', 'Converted race_date to datetime, ensured numeric types', rows_before, len(master))


Data type conversions applied:
resultId                            int64
raceId                              int64
driverId                            int64
constructorId                       int64
grid                                int64
positionOrder                       int64
points                            float64
laps                                int64
statusId                            int64
year                                int64
round                               int64
circuitId                           int64
race_name                          object
race_date                  datetime64[ns]
constructor_name                   object
constructor_nationality            object
forename                           object
surname                            object
driver_nationality                 object
status                             object
circuit_name                       object
circuit_location                   object
circuit_country                    object
lat

## 6. Step 3 : Handle Remaining Nulls
After dropping the sparse columns, let us check if any nulls remain in the critical columns.


In [6]:
# Check remaining nulls
remaining_nulls = master.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

if len(remaining_nulls) == 0:
    print('No remaining nulls in the master table.')
else:
    print('Remaining nulls:')
    print(remaining_nulls.to_string())
    print()
    # Handle any remaining nulls
    for col in remaining_nulls.index:
        null_count = remaining_nulls[col]
        null_pct = null_count / len(master) * 100
        print(f'  {col}: {null_count} nulls ({null_pct:.1f}%) — action: ', end='')
        if null_pct < 1:
            master = master.dropna(subset=[col])
            print(f'dropped {null_count} rows (< 1% of data)')
        else:
            print('kept as-is (too many to drop)')

log_step('Handle Nulls', 'Addressed remaining null values', rows_before, len(master))


No remaining nulls in the master table.
[Handle Nulls] Addressed remaining null values | 4626 -> 4626 rows


## 7. Step 4 : Standardise Text Columns


In [7]:
rows_before = len(master)

# Strip whitespace from all string columns
str_cols = master.select_dtypes(include='object').columns
for col in str_cols:
    master[col] = master[col].str.strip()

# Standardise constructor names (check for inconsistencies)
print('Unique constructor names:')
print(master['constructor_name'].sort_values().unique())
print()
print(f'Total unique constructors: {master["constructor_name"].nunique()}')

log_step('Standardise Text', 'Stripped whitespace from all text columns', rows_before, len(master))


Unique constructor names:
['Alfa Romeo' 'AlphaTauri' 'Alpine F1 Team' 'Aston Martin' 'Caterham'
 'Ferrari' 'Force India' 'Haas F1 Team' 'Lotus F1' 'Manor Marussia'
 'Marussia' 'McLaren' 'Mercedes' 'RB F1 Team' 'Racing Point' 'Red Bull'
 'Renault' 'Sauber' 'Toro Rosso' 'Williams']

Total unique constructors: 20
[Standardise Text] Stripped whitespace from all text columns | 4626 -> 4626 rows


## 8. Step 5 : Create Derived Columns
We engineer new columns that are directly tied to our KPIs:
- **`position_delta`**: Difference between grid (start) and positionOrder (finish). Negative = gained positions.
- **`finished`**: Boolean flag , did the driver finish the race?
- **`driver_name`**: Combined full name for readability.


In [8]:
rows_before = len(master)

# Position Delta (grid - finish): negative means the driver gained positions during the race
master['position_delta'] = master['grid'] - master['positionOrder']

# Finished flag (status == 'Finished' or '+X Lap')
master['finished'] = master['status'].str.contains('Finished|Lap', case=False, na=False).astype(int)

# Combined driver name
master['driver_name'] = master['forename'] + ' ' + master['surname']

print('New columns created:')
print(f'  position_delta: {master["position_delta"].describe().to_string()}')
print()
print(f'  finished distribution: {master["finished"].value_counts().to_string()}')
print()
print(f'  Sample driver names: {master["driver_name"].unique()[:5]}')

log_step('Derived Columns', 'Created position_delta, finished, driver_name', rows_before, len(master))


New columns created:
  position_delta: count   4626.00
mean      -0.31
std        5.45
min      -22.00
25%       -2.00
50%        0.00
75%        3.00
max       19.00

  finished distribution: finished
1    3825
0     801

  Sample driver names: ['Nico Rosberg' 'Kevin Magnussen' 'Jenson Button' 'Fernando Alonso'
 'Valtteri Bottas']
[Derived Columns] Created position_delta, finished, driver_name | 4626 -> 4626 rows


## 9. Step 6 : Remove Duplicates


In [9]:
rows_before = len(master)

# Check for exact duplicates
dupes = master.duplicated().sum()
print(f'Exact duplicate rows found: {dupes}')

if dupes > 0:
    master = master.drop_duplicates()
    print(f'Removed {dupes} duplicate rows.')

# Check for logical duplicates (same driver + same race should be unique)
logical_dupes = master.duplicated(subset=['raceId', 'driverId']).sum()
print(f'Logical duplicates (same driver in same race): {logical_dupes}')

log_step('Duplicates', f'Found {dupes} exact duplicates, {logical_dupes} logical duplicates', rows_before, len(master))


Exact duplicate rows found: 0
Logical duplicates (same driver in same race): 0
[Duplicates] Found 0 exact duplicates, 0 logical duplicates | 4626 -> 4626 rows


## 10. Clean Pit Stops Data
The pit stops table has a different granularity (one row per pit stop, not per race result).  
We clean it separately.


In [10]:
print(f'Pit stops shape: {pit_stops.shape}')
print()

# Check for nulls
print('Null counts:')
print(pit_stops.isna().sum().to_string())
print()

# Convert duration from string to float (some entries are in mm:ss.sss format)
# The 'milliseconds' column is already numeric, so we use that
pit_stops['duration_seconds'] = pit_stops['milliseconds'] / 1000.0

# Flag outliers (pit stops > 60 seconds are likely red flag periods or drive-throughs)
pit_stops['is_normal_stop'] = (pit_stops['duration_seconds'] < 60).astype(int)

print(f'Normal stops (< 60s): {pit_stops["is_normal_stop"].sum():,}')
print(f'Abnormal stops (>= 60s): {(pit_stops["is_normal_stop"] == 0).sum():,}')
print()
print(f'Normal stop stats:')
print(pit_stops[pit_stops['is_normal_stop'] == 1]['duration_seconds'].describe().to_string())


Pit stops shape: (8360, 7)

Null counts:
raceId          0
driverId        0
stop            0
lap             0
time            0
duration        0
milliseconds    0

Normal stops (< 60s): 7,851
Abnormal stops (>= 60s): 509

Normal stop stats:
count   7851.00
mean      24.82
std        4.73
min       13.97
25%       22.19
50%       23.79
75%       26.25
max       59.55


## 11. Clean Constructor Standings
We need the **final standings** at the end of each season for the YoY Growth KPI.  

> That means we take only the last race of each season.


In [11]:
# We need the final race of each season to get year-end standings
# Join with races to get the year
races_year = pd.read_csv('../data/raw/races.csv', na_values=['\\N'])[['raceId', 'year', 'round']]
standings_with_year = standings.merge(races_year, on='raceId', how='left')

# Get the last round of each year
last_rounds = standings_with_year.groupby('year')['round'].max().reset_index()
last_rounds.columns = ['year', 'last_round']

# Filter to only the final race standings
standings_final = standings_with_year.merge(last_rounds, on='year')
standings_final = standings_final[standings_final['round'] == standings_final['last_round']].copy()

# Join constructor names
constructors_df = pd.read_csv('../data/raw/constructors.csv', na_values=['\\N'])[['constructorId', 'name']]
constructors_df = constructors_df.rename(columns={'name': 'constructor_name'})
standings_final = standings_final.merge(constructors_df, on='constructorId', how='left')

# Keep only relevant columns
standings_final = standings_final[['year', 'constructorId', 'constructor_name', 'points', 'position', 'wins']].copy()
standings_final = standings_final.sort_values(['year', 'position']).reset_index(drop=True)

print(f'Year-end constructor standings: {standings_final.shape}')
print()
print(standings_final.head(20))


Year-end constructor standings: (112, 6)

    year  constructorId constructor_name  points  position  wins
0   2014            131         Mercedes  701.00         1    16
1   2014              9         Red Bull  405.00         2     3
2   2014              3         Williams  320.00         3     0
3   2014              6          Ferrari  216.00         4     0
4   2014              1          McLaren  181.00         5     0
5   2014             10      Force India  155.00         6     0
6   2014              5       Toro Rosso   30.00         7     0
7   2014            208         Lotus F1   10.00         8     0
8   2014            206         Marussia    2.00         9     0
9   2014             15           Sauber    0.00        10     0
10  2014            207         Caterham    0.00        11     0
11  2015            131         Mercedes  703.00         1    16
12  2015              6          Ferrari  428.00         2     3
13  2015              3         Williams  257.00

## 12. Post-Cleaning Summary


In [12]:
print('---- CLEANING LOG ----')
print()
log_df = pd.DataFrame(cleaning_log)
display(log_df)

print()
print('---- FINAL CLEANED DATASET ----')
print(f'Shape: {master.shape}')
print(f'Year range: {master["year"].min()} to {master["year"].max()}')
print(f'Total nulls remaining: {master.isna().sum().sum()}')
print(f'Columns: {list(master.columns)}')
print()
print('Column types:')
print(master.dtypes.to_string())


---- CLEANING LOG ----



,Step,Description,Rows Before,Rows After,Rows Removed,Cols Before,Cols After
0,Drop Columns,Removed 10 sparse/irrelevant columns,4626,4626,0,35.00,25.00
1,Fix Dtypes,"Converted race_date to datetime, ensured numer...",4626,4626,0,NaN,NaN
2,Handle Nulls,Addressed remaining null values,4626,4626,0,NaN,NaN
3,Standardise Text,Stripped whitespace from all text columns,4626,4626,0,NaN,NaN
4,Derived Columns,"Created position_delta, finished, driver_name",4626,4626,0,NaN,NaN
5,Duplicates,"Found 0 exact duplicates, 0 logical duplicates",4626,4626,0,NaN,NaN



---- FINAL CLEANED DATASET ----
Shape: (4626, 28)
Year range: 2014 to 2024
Total nulls remaining: 0
Columns: ['resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'points', 'laps', 'statusId', 'year', 'round', 'circuitId', 'race_name', 'race_date', 'constructor_name', 'constructor_nationality', 'forename', 'surname', 'driver_nationality', 'status', 'circuit_name', 'circuit_location', 'circuit_country', 'lat', 'lng', 'position_delta', 'finished', 'driver_name']

Column types:
resultId                            int64
raceId                              int64
driverId                            int64
constructorId                       int64
grid                                int64
positionOrder                       int64
points                            float64
laps                                int64
statusId                            int64
year                                int64
round                               int64
circuitId                         

## 13. Save Cleaned Data to `data/processed/`
These are the final, analysis-ready files that will be used by:
- `03_eda.ipynb` for exploratory analysis
- `04_statistical_analysis.ipynb` for correlation/regression
- `05_final_load_prep.ipynb` for Tableau export


In [13]:
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Save main cleaned dataset
master.to_csv(os.path.join(PROCESSED_DIR, 'f1_modern_clean.csv'), index=False)
print(f'Saved f1_modern_clean.csv ({len(master):,} rows x {master.shape[1]} cols)')

# Save cleaned pit stops
pit_stops.to_csv(os.path.join(PROCESSED_DIR, 'pit_stops_clean.csv'), index=False)
print(f'Saved pit_stops_clean.csv ({len(pit_stops):,} rows x {pit_stops.shape[1]} cols)')

# Save year-end constructor standings
standings_final.to_csv(os.path.join(PROCESSED_DIR, 'constructor_standings_final.csv'), index=False)
print(f'Saved constructor_standings_final.csv ({len(standings_final):,} rows x {standings_final.shape[1]} cols)')

print()
print('Cleaning phase complete. Proceed to 03_eda.ipynb.')


Saved f1_modern_clean.csv (4,626 rows x 28 cols)
Saved pit_stops_clean.csv (8,360 rows x 9 cols)
Saved constructor_standings_final.csv (112 rows x 6 cols)

Cleaning phase complete. Proceed to 03_eda.ipynb.


## 14. Quick Sanity Check (Sample Rows)


In [14]:
# Show a few rows from different years to verify everything looks right
for yr in [2014, 2018, 2022, master['year'].max()]:
    sample = master[master['year'] == yr].head(3)
    if len(sample) > 0:
        print(f'--- {yr} ---')
        display(sample[['year', 'race_name', 'driver_name', 'constructor_name', 'grid', 'positionOrder', 'points', 'status', 'position_delta', 'finished']])
        print()


--- 2014 ---


,year,race_name,driver_name,constructor_name,grid,positionOrder,points,status,position_delta,finished
0,2014,Australian Grand Prix,Nico Rosberg,Mercedes,3,1,25.00,Finished,2,1
1,2014,Australian Grand Prix,Kevin Magnussen,McLaren,4,2,18.00,Finished,2,1
2,2014,Australian Grand Prix,Jenson Button,McLaren,10,3,15.00,Finished,7,1



--- 2018 ---


,year,race_name,driver_name,constructor_name,grid,positionOrder,points,status,position_delta,finished
1647,2018,Australian Grand Prix,Sebastian Vettel,Ferrari,3,1,25.00,Finished,2,1
1648,2018,Australian Grand Prix,Lewis Hamilton,Mercedes,1,2,18.00,Finished,-1,1
1649,2018,Australian Grand Prix,Kimi Räikkönen,Ferrari,2,3,15.00,Finished,-1,1



--- 2022 ---


,year,race_name,driver_name,constructor_name,grid,positionOrder,points,status,position_delta,finished
3267,2022,Bahrain Grand Prix,Charles Leclerc,Ferrari,1,1,26.00,Finished,0,1
3268,2022,Bahrain Grand Prix,Carlos Sainz,Ferrari,3,2,18.00,Finished,1,1
3269,2022,Bahrain Grand Prix,Lewis Hamilton,Mercedes,5,3,15.00,Finished,2,1



--- 2024 ---


,year,race_name,driver_name,constructor_name,grid,positionOrder,points,status,position_delta,finished
4147,2024,Bahrain Grand Prix,Max Verstappen,Red Bull,1,1,26.00,Finished,0,1
4148,2024,Bahrain Grand Prix,Sergio Pérez,Red Bull,5,2,18.00,Finished,3,1
4149,2024,Bahrain Grand Prix,Carlos Sainz,Ferrari,4,3,15.00,Finished,1,1
